# Casualty Cat Model Walkthrough

This notebook walks through the full modeling pipeline: setting up a scenario, running a Monte Carlo simulation, computing standard cat model outputs, and quantifying tail uncertainty.

The agent layer is demonstrated in a separate section at the end.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import matplotlib.pyplot as plt

from cat_model import (
    FrequencyParams, SeverityParams, build_sample_portfolio,
    SimulationConfig, run_simulation, bootstrap_metric_ci,
    one_at_a_time, tornado,
)

np.random.seed(0)

## 1. Define the scenario

We model a stylized PFAS-style emerging mass tort:

- **Frequency:** ~1 event every 6-7 years on average, with overdispersion (events cluster — once one verdict lands, copycat suits follow)
- **Severity:** $50M mean per event, lognormal with CV=2.5 (heavy-tailed)
- **Portfolio:** 50 policies across chemicals/pharma/consumer-products, $500M total limit

In [ ]:
frequency = FrequencyParams(mean=0.15, dispersion=2.0)
severity = SeverityParams.from_mean_cv(mean=50_000_000, cv=2.5)
portfolio = build_sample_portfolio(n_policies=50, total_limit=500_000_000, seed=42)

print(f'Frequency: {frequency.mean:.2f} events/year, dispersion {frequency.dispersion:.1f}')
print(f'Severity:  mean ${severity.mean:,.0f}, sigma {severity.sigma:.2f}')
print(f'Portfolio: {portfolio.n_policies} policies, ${portfolio.total_limit:,.0f} total limit')

## 2. Run the simulation

100,000 simulated years. Each year draws an event count from the Negative Binomial frequency distribution, then severities for each event from the lognormal, applies contract terms, and aggregates to an annual portfolio loss.

In [ ]:
config = SimulationConfig(
    frequency=frequency,
    severity=severity,
    portfolio=portfolio,
    n_years=100_000,
    seed=42,
)
result = run_simulation(config)

print(f'AAL:        ${result.aal:>15,.0f}  (Monte Carlo SE: ${result.aal_std_error:,.0f})')
print(f'100-yr PML: ${result.pml(100):>15,.0f}')
print(f'250-yr PML: ${result.pml(250):>15,.0f}')
print(f'500-yr PML: ${result.pml(500):>15,.0f}')
print(f'250-yr TVaR:${result.tvar(250):>15,.0f}')
print(f'\nLoss-free years: {(result.annual_losses == 0).mean()*100:.1f}%')

## 3. The exceedance probability curve

The classic cat model output. Vertical axis: annual loss; horizontal: return period. The shape of the tail tells you about capital adequacy.

In [ ]:
rps, pmls = result.ep_curve(np.array([2, 5, 10, 25, 50, 100, 250, 500, 1000]))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rps, pmls / 1e6, 'o-', linewidth=2)
ax.set_xscale('log')
ax.set_xlabel('Return period (years, log scale)')
ax.set_ylabel('PML ($M)')
ax.set_title('Exceedance Probability Curve')
ax.grid(True, alpha=0.3)
for rp, pml in zip(rps, pmls):
    ax.annotate(f'${pml/1e6:.0f}M', (rp, pml/1e6), textcoords='offset points', xytext=(5, 5), fontsize=8)
plt.tight_layout()
plt.show()

## 4. Quantifying tail uncertainty

A 250-year PML from 100,000 simulated years samples roughly 400 tail events. That sounds like a lot, but the bootstrap CI tells us how reliable the point estimate actually is.

In [ ]:
for rp in [100, 250, 500, 1000]:
    q = 1 - 1.0 / rp
    point, lo, hi = bootstrap_metric_ci(
        result.annual_losses,
        metric_fn=lambda x: float(np.quantile(x, q)),
        n_bootstrap=1000,
        seed=0,
    )
    width_pct = (hi - lo) / point * 100 if point > 0 else 0
    print(f'{rp:>4}-yr PML: ${point/1e6:>6.1f}M   95% CI [${lo/1e6:>6.1f}M, ${hi/1e6:>6.1f}M]   width: {width_pct:.1f}%')

Notice the CI width *grows* as we go deeper into the tail — the 1000-year PML is the most uncertain. This is exactly why honest tail estimation requires either more sims or analytical extreme-value methods.

## 5. Sensitivity: one-at-a-time

How does the 250-year PML respond to perturbations in each parameter?

In [ ]:
small_cfg = SimulationConfig(
    frequency=frequency, severity=severity, portfolio=portfolio,
    n_years=20_000, seed=42,
)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
params = ['frequency_mean', 'frequency_dispersion', 'severity_mean', 'severity_cv']
for ax, param in zip(axes.flat, params):
    points = one_at_a_time(small_cfg, param, multipliers=np.array([0.5, 0.75, 1.0, 1.25, 1.5]))
    mults = [p.multiplier for p in points]
    pmls = [p.pml_250yr / 1e6 for p in points]
    ax.plot(mults, pmls, 'o-', linewidth=2)
    ax.axhline(pmls[2], color='gray', linestyle='--', alpha=0.5)
    ax.set_title(param)
    ax.set_xlabel('multiplier')
    ax.set_ylabel('250-yr PML ($M)')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Tornado analysis: which assumption matters most?

In [ ]:
rows = tornado(small_cfg, shock=0.25, metric='pml_250yr')

fig, ax = plt.subplots(figsize=(9, 4))
names = [r['parameter'] for r in rows]
lows = [r['low_pct_change'] for r in rows]
highs = [r['high_pct_change'] for r in rows]
y = np.arange(len(names))
ax.barh(y, highs, color='tab:red', alpha=0.7, label='+25%')
ax.barh(y, lows, color='tab:blue', alpha=0.7, label='-25%')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_yticks(y)
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel('% change in 250-yr PML')
ax.set_title('Tornado: Drivers of 250-year PML')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

for r in rows:
    print(f"  {r['parameter']:>22s}  -25%: {r['low_pct_change']:+6.1f}%  +25%: {r['high_pct_change']:+6.1f}%")

## 7. The agent layer

All of the above can be invoked through natural-language queries. Set your `OPENAI_API_KEY` and uncomment the cell below.

In [ ]:
# from agent import CatModelAgent, ModelContext, trace_summary
# 
# ctx = ModelContext(default_n_years=20_000)
# agent = CatModelAgent(ctx=ctx, model='gpt-4o-mini')
# 
# response = agent.run("What's the 250-year PML and how confident should I be in that estimate?")
# print(response.final_text)
# print('\n--- trace ---')
# print(trace_summary(response.trace))